In [0]:
# ============================================
# NOTEBOOK 3: Gold Layer - Analytics (Fixed)
# PROJECT: Wikipedia Clickstream Analytics
# ============================================
from pyspark.sql.functions import col, sum, desc, count, hour, round as spark_round

# Load silver tables
silver_clickstream = spark.table("workspace.default.silver_clickstream") \
    .withColumn("hour_of_day", hour(col("ingested_at")))
silver_pageviews = spark.table("workspace.default.silver_pageviews") \
    .withColumn("hour_of_day", hour(col("ingested_at")))

print("Silver Clickstream count:", silver_clickstream.count())
print("Silver Pageviews count:", silver_pageviews.count())
print("✅ Silver tables loaded!")

# ── Q1: Top Pages Per Hour ──
gold_top_pages = silver_pageviews \
    .groupBy("hour_of_day", "article_title") \
    .agg(sum("view_count").alias("total_views")) \
    .orderBy(desc("total_views")) \
    .limit(10)
gold_top_pages.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.gold_top_pages")
print("✅ Q1 gold_top_pages saved!")

# ── Q2: Entry Points ──
gold_entry_points = silver_clickstream \
    .filter(col("click_type") == "other") \
    .groupBy("target_page") \
    .agg(sum("click_count").alias("total_entries")) \
    .orderBy(desc("total_entries")) \
    .limit(10)
gold_entry_points.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.gold_entry_points")
print("✅ Q2 gold_entry_points saved!")

# ── Q3: Exit Points with Ratio ──
total_exits = silver_clickstream \
    .groupBy("source_page") \
    .agg(sum("click_count").alias("total_exits"))
total_entries_df = silver_clickstream \
    .groupBy("target_page") \
    .agg(sum("click_count").alias("total_entries"))
gold_exit_points = total_exits.join(
    total_entries_df,
    total_exits.source_page == total_entries_df.target_page,
    "left"
).fillna(0) \
.withColumn("exit_ratio",
    spark_round(col("total_exits") / (col("total_exits") + col("total_entries")), 4)) \
.select("source_page", "total_exits", "total_entries", "exit_ratio") \
.orderBy(desc("exit_ratio")) \
.limit(10)
gold_exit_points.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.gold_exit_points")
print("✅ Q3 gold_exit_points saved!")

# ── Q4: Click Paths A→B→C ──
hop1 = silver_clickstream.select(
    col("source_page").alias("page_a"),
    col("target_page").alias("page_b"),
    col("click_count").alias("count1"))
hop2 = silver_clickstream.select(
    col("source_page").alias("page_b2"),
    col("target_page").alias("page_c"),
    col("click_count").alias("count2"))
gold_click_paths = hop1.join(hop2, hop1.page_b == hop2.page_b2) \
    .select(
        col("page_a"), col("page_b"), col("page_c"),
        (col("count1") + col("count2")).alias("total_clicks")) \
    .orderBy(desc("total_clicks")) \
    .limit(10)
gold_click_paths.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.gold_click_paths")
print("✅ Q4 gold_click_paths saved!")

# ── Q5: Hourly Traffic ──
gold_hourly_traffic = silver_pageviews \
    .groupBy("hour_of_day") \
    .agg(
        sum("view_count").alias("total_views"),
        count("article_title").alias("unique_articles")) \
    .orderBy("hour_of_day")
gold_hourly_traffic.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.gold_hourly_traffic")
print("✅ Q5 gold_hourly_traffic saved!")

# ── Q6: Click Types with Percentage ──
total_clicks_global = silver_clickstream \
    .agg(sum("click_count")).collect()[0][0]
gold_click_types = silver_clickstream \
    .groupBy("click_type") \
    .agg(
        sum("click_count").alias("total_clicks"),
        count("source_page").alias("num_connections")) \
    .withColumn("percentage",
        spark_round((col("total_clicks") / total_clicks_global) * 100, 2)) \
    .orderBy(desc("total_clicks"))
gold_click_types.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.gold_click_types")
print("✅ Q6 gold_click_types saved!")

print("\n🎉 ALL 6 GOLD TABLES SAVED SUCCESSFULLY!")

Silver Clickstream count: 24522123
Silver Pageviews count: 861793
✅ Silver tables loaded!
✅ Q1 gold_top_pages saved!
✅ Q2 gold_entry_points saved!
✅ Q3 gold_exit_points saved!
✅ Q4 gold_click_paths saved!
✅ Q5 gold_hourly_traffic saved!
✅ Q6 gold_click_types saved!

🎉 ALL 6 GOLD TABLES SAVED SUCCESSFULLY!
